In [2]:
from datasets import load_dataset
import torch
from torch.utils.data import DataLoader, TensorDataset
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW, AutoModelForMaskedLM, AutoModelForMaskedLM, DataCollatorForLanguageModeling
from transformers import BertForSequenceClassification
import torch
from transformers import get_scheduler
from transformers import Trainer
from transformers import BertTokenizer
import numpy as np
import pandas as pd
from transformers import BertForSequenceClassification, get_linear_schedule_with_warmup
from transformers import BertPreTrainedModel
from torch import nn
from transformers import BertModel
from torch.nn import CrossEntropyLoss
from transformers.modeling_outputs import SequenceClassifierOutput

#os.environ["http_proxy"] = "http://127.0.0.1:10809"
#os.environ["https_proxy"] = "http://127.0.0.1:10809"
# os.environ["CUDA_VISIBLE_DEVICES"] = "2"

tokenizer = BertTokenizer.from_pretrained("G:/Other computers/我的计算机/cloud_share/Job_posting_data/chinese-bert-wwm/")


c:\Users\DELL\anaconda3\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.2
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [9]:
df = pd.read_csv("F:/Data/job_posting/processed/finetune/est_sample.csv", encoding = "utf_8_sig", on_bad_lines='skip', encoding_errors='ignore')

In [99]:
# replace the symbol '-' to '.' in soc_code column, and convert soc_code to int
df['soc_code'] = df['soc_code'].str.replace('-', '')

# create a new column 'major_group' as the first two digits of soc_code
df['major_group'] = df['soc_code'].str[:2].astype(int)

# create a new column 'minor_group' as the third and four digits of soc_code
df['minor_group'] = df['soc_code'].str[2:4].astype(int)

# create a new column 'broad_group' as the first six digits of soc_code
df['broad_group'] = df['soc_code'].str[4:6].astype(int)

# generate a new column 'soc_code1' with value to recode the 'soc_code' in ascending order
df['major_group1'] = df['major_group'].rank(method='dense').astype(int) - 1

# Assign new values to 'minor_group' within each 'major_group1' in ascending order
df['minor_group1'] = df.groupby('major_group1')['minor_group'].rank(method='dense').astype(int) - 1

# Assign new values to 'broad_group' within each 'major_group1' and 'minor_group1' in ascending order
df['broad_group1'] = df.groupby(['major_group1', 'minor_group1'])['broad_group'].rank(method='dense').astype(int) - 1

In [100]:
# create into train and test set
from sklearn.model_selection import train_test_split
train_df_sample, test_df_sample = train_test_split(df, test_size=0.2, random_state=42)

# drop index column, 'true_ind' and 'sample' columns
train_df_sample = train_df_sample.drop(['Unnamed: 0'], axis = 1)

# drop index column, 'true_ind' and 'sample' columns
test_df_sample = test_df_sample.drop(['Unnamed: 0'], axis = 1)


In [101]:
# convert '工作描述' to string
train_df_sample['工作描述'] = train_df_sample['工作描述'].astype(str)
test_df_sample['工作描述'] = test_df_sample['工作描述'].astype(str)

In [102]:
# Tokenize the text and convert it into input features
train_texts = train_df_sample['工作描述'].tolist()
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=512)

test_texts = test_df_sample['工作描述'].tolist()
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=512)

KeyboardInterrupt: 

In [ ]:
# Convert the input features into PyTorch tensors
train_inputs = torch.tensor(train_encodings['input_ids'])
train_masks = torch.tensor(train_encodings['attention_mask'])

test_inputs = torch.tensor(test_encodings['input_ids'])
test_masks = torch.tensor(test_encodings['attention_mask'])

train_major_labels = torch.tensor(train_df_sample['major_group1'].tolist())
train_minor_labels = torch.tensor(train_df_sample['minor_group1'].tolist())
train_broad_labels = torch.tensor(train_df_sample['broad_group1'].tolist())

test_major_labels = torch.tensor(test_df_sample['major_group1'].tolist())
test_minor_labels = torch.tensor(test_df_sample['minor_group1'].tolist())
test_broad_labels = torch.tensor(test_df_sample['broad_group1'].tolist())


### Update your TensorDataset and DataLoader objects to include the new label tensors

In [ ]:
# Create a PyTorch DataLoader to iterate over the data during training
batch_size = 5

train_data = TensorDataset(train_inputs, train_masks, train_major_labels, train_minor_labels, train_broad_labels)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=5)

test_data = TensorDataset(test_inputs, test_masks, test_major_labels, test_minor_labels, test_broad_labels)
test_loader = DataLoader(test_data, batch_size=batch_size, num_workers=5)


### Create a custom BERT model that outputs three separate logits for each level of the hierarchy

In [ ]:
from transformers import BertPreTrainedModel
from torch import nn
from transformers import BertModel

class HierarchicalBert(BertPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.bert = BertModel(config)
        self.major_classifier = nn.Linear(config.hidden_size, num_major_labels)
        self.minor_classifier = nn.Linear(config.hidden_size, num_minor_labels)
        self.broad_classifier = nn.Linear(config.hidden_size, num_broad_labels)

    def forward(self, input_ids, attention_mask, major_labels=None, minor_labels=None, broad_labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output

        major_logits = self.major_classifier(pooled_output)
        minor_logits = self.minor_classifier(pooled_output)
        broad_logits = self.broad_classifier(pooled_output)


        return major_logits, minor_logits, broad_logits


### Instantiate the custom model and update the optimizer

In [ ]:
num_epochs = 4
num_training_steps = num_epochs * len(train_loader)


num_major_labels = len(train_df_sample['major_group1'].unique())
num_minor_labels = len(train_df_sample['minor_group1'].unique())
num_broad_labels = len(train_df_sample['broad_group1'].unique())

# Create separate BERT models for each level of the hierarchy
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move the data and labels to the same device as the models
train_inputs = train_inputs.to(device)
train_masks = train_masks.to(device)
train_major_labels = train_major_labels.to(device)
train_minor_labels = train_minor_labels.to(device)
train_broad_labels = train_broad_labels.to(device)

test_inputs = test_inputs.to(device)
test_masks = test_masks.to(device)
test_major_labels = test_major_labels.to(device)
test_minor_labels = test_minor_labels.to(device)
test_broad_labels = test_broad_labels.to(device)


major_model = BertForSequenceClassification.from_pretrained("G:/Other computers/我的计算机/cloud_share/Job_posting_data/chinese-bert-wwm/", num_labels=num_major_labels).to(device)
minor_model = BertForSequenceClassification.from_pretrained("G:/Other computers/我的计算机/cloud_share/Job_posting_data/chinese-bert-wwm/", num_labels=num_minor_labels).to(device)
broad_model = BertForSequenceClassification.from_pretrained("G:/Other computers/我的计算机/cloud_share/Job_posting_data/chinese-bert-wwm/", num_labels=num_broad_labels).to(device)


# Create separate optimizers and schedulers for each model
major_optimizer = AdamW(major_model.parameters(), lr=2e-5, eps=1e-8)
minor_optimizer = AdamW(minor_model.parameters(), lr=2e-5, eps=1e-8)
broad_optimizer = AdamW(broad_model.parameters(), lr=2e-5, eps=1e-8)

major_scheduler = get_linear_schedule_with_warmup(major_optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)
minor_scheduler = get_linear_schedule_with_warmup(minor_optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)
broad_scheduler = get_linear_schedule_with_warmup(broad_optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

### Update the training loop to handle the hierarchical outputs and compute the loss for each level

In [ ]:
criterion = nn.CrossEntropyLoss()

# Train the major model
# Train the major model
for epoch in range(num_epochs):
    major_model.train()
    total_loss = 0
    num_batches = 0

    for batch in train_loader:
        inputs = batch[0].to(device)
        masks = batch[1].to(device)
        major_labels = batch[2].to(device)

        major_optimizer.zero_grad()

        outputs = major_model(inputs, attention_mask=masks, labels=major_labels)
        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(major_model.parameters(), 1.0)

        major_optimizer.step()
        major_scheduler.step()

        total_loss += loss.item()
        num_batches += 1

    epoch_loss = total_loss / num_batches
    print(f"Major Model - Epoch {epoch + 1}/{num_epochs}, Loss: {epoch_loss:.4f}")

# Train the minor model, using major model predictions
# Train the minor model
for epoch in range(num_epochs):
    minor_model.train()
    total_loss = 0
    num_batches = 0

    for batch in train_loader:
        inputs = batch[0].to(device)
        masks = batch[1].to(device)
        major_labels = batch[2].to(device)
        minor_labels = batch[3].to(device)

        with torch.no_grad():
            major_outputs = major_model(inputs, attention_mask=masks)
            major_predictions = torch.argmax(major_outputs.logits, axis=1)

        filtered_indices = torch.where(major_predictions.cpu() == major_labels)[0]
        if len(filtered_indices) == 0:
            continue

        filtered_inputs = inputs[filtered_indices]
        filtered_masks = masks[filtered_indices]
        filtered_minor_labels = minor_labels[filtered_indices]

        minor_optimizer.zero_grad()

        outputs = minor_model(filtered_inputs, attention_mask=filtered_masks, labels=filtered_minor_labels)
        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(minor_model.parameters(), 1.0)

        minor_optimizer.step()
        minor_scheduler.step()

        total_loss += loss.item()
        num_batches += 1

    epoch_loss = total_loss / num_batches
    print(f"Minor Model - Epoch {epoch + 1}/{num_epochs}, Loss: {epoch_loss:.4f}")


# Train the broad model, using major and minor model predictions
# Train the broad model
for epoch in range(num_epochs):
    broad_model.train()
    total_loss = 0
    num_batches = 0

    for batch in train_loader:
        inputs = batch[0].to(device)
        masks = batch[1].to(device)
        major_labels = batch[2].to(device)
        minor_labels = batch[3].to(device)
        broad_labels = batch[4].to(device)

        with torch.no_grad():
            major_outputs = major_model(inputs, attention_mask=masks)
            major_predictions = torch.argmax(major_outputs.logits, axis=1)
            minor_outputs = minor_model(inputs, attention_mask=masks)
            minor_predictions = torch.argmax(minor_outputs.logits, axis=1)

        filtered_indices = torch.where((major_predictions.cpu() == major_labels) & (minor_predictions.cpu() == minor_labels))[0]
        if len(filtered_indices) == 0:
            continue

        filtered_inputs = inputs[filtered_indices]
        filtered_masks = masks[filtered_indices]
        filtered_broad_labels = broad_labels[filtered_indices]

        broad_optimizer.zero_grad()

        outputs = broad_model(filtered_inputs, attention_mask=filtered_masks, labels=filtered_broad_labels)
        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(broad_model.parameters(), 1.0)

        broad_optimizer.step()
        broad_scheduler.step()

        total_loss += loss.item()
        num_batches += 1

    epoch_loss = total_loss / num_batches
    print(f"Broad Model - Epoch {epoch + 1}/{num_epochs}, Loss: {epoch_loss:.4f}")


### Update evaluation code as well to handle the three separate outputs and compute evaluation metrics for each level of the hierarchy

- Evaluation function calculates the accuracy independently for each level.

In [ ]:
# Evaluate the performance of the model on the test set
def evaluate(model, test_loader, level):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in test_loader:
            inputs = batch[0].to(device)
            masks = batch[1].to(device)
            labels = batch[level + 1].to(device)  # major_labels: 2, minor_labels: 3, broad_labels: 4

            outputs = model(inputs, attention_mask=masks)
            _, predicted = torch.max(outputs.logits, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = correct / total
    return accuracy


# Evaluate major model
major_accuracy = evaluate(major_model, test_loader, 1)
print(f"Major Model Accuracy: {major_accuracy:.4f}")

# Evaluate minor model
minor_accuracy = evaluate(minor_model, test_loader, 2)
print(f"Minor Model Accuracy: {minor_accuracy:.4f}")

# Evaluate broad model
broad_accuracy = evaluate(broad_model, test_loader, 3)
print(f"Broad Model Accuracy: {broad_accuracy:.4f}")


-  The end-to-end performance refers to the evaluation of the whole system, from the input (job descriptions) to the final output (correctly predicted major, minor, and broad labels) instead of evaluating the accuracy of each level separately.

In [ ]:
def evaluate_end_to_end(major_model, minor_model, broad_model, test_loader):
    major_model.eval()
    minor_model.eval()
    broad_model.eval()
    
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in test_loader:
            inputs = batch[0].to(device)
            masks = batch[1].to(device)
            major_labels = batch[2].to(device)
            minor_labels = batch[3].to(device)
            broad_labels = batch[4].to(device)

            major_outputs = major_model(inputs, attention_mask=masks)
            major_predictions = torch.argmax(major_outputs.logits, axis=1)

            minor_outputs = minor_model(inputs, attention_mask=masks)
            minor_predictions = torch.argmax(minor_outputs.logits, axis=1)

            broad_outputs = broad_model(inputs, attention_mask=masks)
            broad_predictions = torch.argmax(broad_outputs.logits, axis=1)

            total += major_labels.size(0)
            correct += ((major_predictions == major_labels) & (minor_predictions == minor_labels) & (broad_predictions == broad_labels)).sum().item()

    accuracy = correct / total
    return accuracy


end_to_end_accuracy = evaluate_end_to_end(major_model, minor_model, broad_model, test_loader)
print(f"End-to-End Accuracy: {end_to_end_accuracy:.4f}")
